In [1]:
# import random
# from itertools import combinations
# from collections import Counter

# # ---- Card representation ----
# RANKS = list(range(2, 15))  # 2-14, 14 = Ace
# SUITS = ['s', 'h', 'd', 'c']
# DECK = [(r, s) for r in RANKS for s in SUITS]

# HIGH_RANKS = {13, 14}  # Kings and Aces

# # ---- Hand evaluation ----

# def best_hand_rank(cards):
#     best = None
#     for combo in combinations(cards, min(5, len(cards))):
#         rank = hand_rank(combo)
#         if best is None or rank > best:
#             best = rank
#     return best

# def hand_rank(cards):
#     ranks = sorted([r for r, s in cards], reverse=True)
#     suits = [s for r, s in cards]
#     counts = Counter(ranks)
#     rank_counts = sorted(counts.values(), reverse=True)

#     is_flush = len(set(suits)) == 1
#     is_straight = (len(set(ranks)) == 5 and ranks[0] - ranks[4] == 4)

#     if set(ranks) == {14, 2, 3, 4, 5}:
#         is_straight = True
#         ranks = [5, 4, 3, 2, 1]
#         counts = Counter(ranks)

#     tiebreaker = sorted(ranks, key=lambda r: (counts[r], r), reverse=True)

#     if is_straight and is_flush:
#         return (8, tiebreaker)
#     if rank_counts[0] == 4:
#         return (7, tiebreaker)
#     if rank_counts[:2] == [3, 2]:
#         return (6, tiebreaker)
#     if is_flush:
#         return (5, tiebreaker)
#     if is_straight:
#         return (4, tiebreaker)
#     if rank_counts[0] == 3:
#         return (3, tiebreaker)
#     if rank_counts[:2] == [2, 2]:
#         return (2, tiebreaker)
#     if rank_counts[0] == 2:
#         return (1, tiebreaker)
#     return (0, tiebreaker)

# # ---- Strategy ----

# def get_rule(pick_number, my_hand):
#     """
#     Picks 1-3: take any card.
#     After pick 3, count how many kings/aces we have:
#       - 0 high cards: picks 4 and 5 target kings/aces only
#       - exactly 1 high card: pick 4 is random, pick 5 targets kings/aces
#       - 2+ high cards: picks 4 and 5 are random
#     """
#     if pick_number <= 3:
#         return lambda c: True  # any card

#     high_card_count = sum(1 for r, s in my_hand if r in HIGH_RANKS)

#     if high_card_count == 0:
#         # No high cards after 3 picks — target kings/aces for both remaining picks
#         return lambda c: c[0] in HIGH_RANKS

#     elif high_card_count == 1:
#         if pick_number == 4:
#             return lambda c: True  # one more random
#         else:  # pick 5
#             return lambda c: c[0] in HIGH_RANKS  # then lock to high cards

#     else:
#         # 2+ high cards already — just take anything
#         return lambda c: True

# # ---- Single game ----

# def play_game(n_picks=5, dealer_fill=8):
#     my_hand = []
#     dealer_hand = []
#     remaining_deck = list(DECK)

#     for pick_number in range(1, n_picks + 1):
#         if not remaining_deck:
#             break

#         rule_fn = get_rule(pick_number, my_hand)

#         matching = [c for c in remaining_deck if rule_fn(c)]
#         if not matching:
#             break

#         random.shuffle(remaining_deck)
#         target_idx = next(i for i, c in enumerate(remaining_deck) if rule_fn(c))

#         my_hand.append(remaining_deck[target_idx])
#         dealer_hand += remaining_deck[:target_idx]
#         remaining_deck = remaining_deck[target_idx + 1:]

#     if len(my_hand) < 5:
#         return 0

#     random.shuffle(remaining_deck)
#     needed = max(0, dealer_fill - len(dealer_hand))
#     final_dealer = dealer_hand + remaining_deck[:needed]

#     return 1 if best_hand_rank(my_hand) > best_hand_rank(final_dealer) else 0

# # ---- Simulate ----

# def simulate(n_games=5000, n_picks=5, dealer_fill=8):
#     wins = sum(play_game(n_picks, dealer_fill) for _ in range(n_games))
#     return wins / n_games

# if __name__ == '__main__':
#     n_games = 5000
#     print("Strategy:")
#     print("  Picks 1-3: any card")
#     print("  If 0 high cards (K/A) after pick 3: picks 4+5 target K/A only")
#     print("  If 1 high card after pick 3:         pick 4 random, pick 5 targets K/A")
#     print("  If 2+ high cards after pick 3:       picks 4+5 random")
#     print(f"\nSimulating over {n_games} games...\n")
#     win_rate = simulate(n_games=n_games)
#     print(f"Win rate: {win_rate:.3f}")

with dealer hand length tracking

In [2]:
# import random
# from itertools import combinations
# from collections import Counter

# # ---- Card representation ----
# RANKS = list(range(2, 15))  # 2-14, 14 = Ace
# SUITS = ['s', 'h', 'd', 'c']
# DECK = [(r, s) for r in RANKS for s in SUITS]
# HIGH_RANKS = {13, 14}  # Kings and Aces

# # ---- Hand evaluation ----

# def best_hand_rank(cards):
#     best = None
#     for combo in combinations(cards, min(5, len(cards))):
#         rank = hand_rank(combo)
#         if best is None or rank > best:
#             best = rank
#     return best

# def hand_rank(cards):
#     ranks = sorted([r for r, s in cards], reverse=True)
#     suits = [s for r, s in cards]
#     counts = Counter(ranks)
#     rank_counts = sorted(counts.values(), reverse=True)
#     is_flush = len(set(suits)) == 1
#     is_straight = (len(set(ranks)) == 5 and ranks[0] - ranks[4] == 4)
#     if set(ranks) == {14, 2, 3, 4, 5}:
#         is_straight = True
#         ranks = [5, 4, 3, 2, 1]
#         counts = Counter(ranks)
#     tiebreaker = sorted(ranks, key=lambda r: (counts[r], r), reverse=True)
#     if is_straight and is_flush:
#         return (8, tiebreaker)
#     if rank_counts[0] == 4:
#         return (7, tiebreaker)
#     if rank_counts[:2] == [3, 2]:
#         return (6, tiebreaker)
#     if is_flush:
#         return (5, tiebreaker)
#     if is_straight:
#         return (4, tiebreaker)
#     if rank_counts[0] == 3:
#         return (3, tiebreaker)
#     if rank_counts[:2] == [2, 2]:
#         return (2, tiebreaker)
#     if rank_counts[0] == 2:
#         return (1, tiebreaker)
#     return (0, tiebreaker)

# # ---- Strategy ----

# def get_rule(pick_number, my_hand):
#     """
#     Picks 1-3: take any card.
#     After pick 3, count how many kings/aces we have:
#       - 0 high cards: picks 4 and 5 target kings/aces only
#       - exactly 1 high card: pick 4 is random, pick 5 targets kings/aces
#       - 2+ high cards: picks 4 and 5 are random
#     """
#     if pick_number <= 3:
#         return lambda c: True
#     high_card_count = sum(1 for r, s in my_hand if r in HIGH_RANKS)
#     if high_card_count == 0:
#         return lambda c: c[0] in HIGH_RANKS
#     elif high_card_count == 1:
#         if pick_number == 4:
#             return lambda c: True
#         else:
#             return lambda c: c[0] in HIGH_RANKS
#     else:
#         return lambda c: True

# # ---- Single game ----

# def play_game(n_picks=5, dealer_fill=8):
#     my_hand = []
#     dealer_hand = []
#     remaining_deck = list(DECK)

#     for pick_number in range(1, n_picks + 1):
#         if not remaining_deck:
#             break
#         rule_fn = get_rule(pick_number, my_hand)
#         matching = [c for c in remaining_deck if rule_fn(c)]
#         if not matching:
#             break
#         random.shuffle(remaining_deck)
#         target_idx = next(i for i, c in enumerate(remaining_deck) if rule_fn(c))
#         my_hand.append(remaining_deck[target_idx])
#         dealer_hand += remaining_deck[:target_idx]
#         remaining_deck = remaining_deck[target_idx + 1:]

#     if len(my_hand) < 5:
#         return 0, 0, 0

#     dealer_size_before_fill = len(dealer_hand)
#     random.shuffle(remaining_deck)
#     needed = max(0, dealer_fill - len(dealer_hand))
#     final_dealer = dealer_hand + remaining_deck[:needed]
#     dealer_size_after_fill = len(final_dealer)

#     win = 1 if best_hand_rank(my_hand) > best_hand_rank(final_dealer) else 0
#     return win, dealer_size_before_fill, dealer_size_after_fill

# # ---- Simulate ----

# def simulate(n_games=5000, n_picks=5, dealer_fill=8):
#     wins = 0
#     total_before = 0
#     total_after = 0
#     for _ in range(n_games):
#         win, before, after = play_game(n_picks, dealer_fill)
#         wins += win
#         total_before += before
#         total_after += after
#     return wins / n_games, total_before / n_games, total_after / n_games

# if __name__ == '__main__':
#     n_games = 5000
#     print("Strategy:")
#     print("  Picks 1-3: any card")
#     print("  If 0 high cards (K/A) after pick 3: picks 4+5 target K/A only")
#     print("  If 1 high card after pick 3:         pick 4 random, pick 5 targets K/A")
#     print("  If 2+ high cards after pick 3:       picks 4+5 random")
#     print(f"\nSimulating over {n_games} games...\n")
#     win_rate, avg_before, avg_after = simulate(n_games=n_games)
#     print(f"Win rate:                          {win_rate:.3f}")
#     print(f"Avg dealer cards after fill to {8}:  {avg_after:.2f}")

fix pick 4

In [3]:
import random
from itertools import combinations
from collections import Counter

# ---- Card representation ----
RANKS = list(range(2, 15))  # 2-14, 14 = Ace
SUITS = ['s', 'h', 'd', 'c']
DECK = [(r, s) for r in RANKS for s in SUITS]
HIGH_RANKS = {13, 14}  # Kings and Aces

# ---- Hand evaluation ----

def best_hand_rank(cards):
    best = None
    for combo in combinations(cards, min(5, len(cards))):
        rank = hand_rank(combo)
        if best is None or rank > best:
            best = rank
    return best

def hand_rank(cards):
    ranks = sorted([r for r, s in cards], reverse=True)
    suits = [s for r, s in cards]
    counts = Counter(ranks)
    rank_counts = sorted(counts.values(), reverse=True)
    is_flush = len(set(suits)) == 1
    is_straight = (len(set(ranks)) == 5 and ranks[0] - ranks[4] == 4)
    if set(ranks) == {14, 2, 3, 4, 5}:
        is_straight = True
        ranks = [5, 4, 3, 2, 1]
        counts = Counter(ranks)
    tiebreaker = sorted(ranks, key=lambda r: (counts[r], r), reverse=True)
    if is_straight and is_flush:
        return (8, tiebreaker)
    if rank_counts[0] == 4:
        return (7, tiebreaker)
    if rank_counts[:2] == [3, 2]:
        return (6, tiebreaker)
    if is_flush:
        return (5, tiebreaker)
    if is_straight:
        return (4, tiebreaker)
    if rank_counts[0] == 3:
        return (3, tiebreaker)
    if rank_counts[:2] == [2, 2]:
        return (2, tiebreaker)
    if rank_counts[0] == 2:
        return (1, tiebreaker)
    return (0, tiebreaker)

# ---- Strategy ----

def get_rule(pick_number, my_hand):
    """
    Picks 1-3: take any card.
    After pick 3, count how many kings/aces we have:
      - 0 high cards: picks 4 and 5 target kings/aces only
      - exactly 1 high card:
          pick 4 is random
          if pick 4 was a K/A -> pick 5 is random
          if pick 4 was not K/A -> pick 5 targets K/A
      - 2+ high cards: picks 4 and 5 are random
    """
    if pick_number <= 3:
        return lambda c: True

    high_card_count = sum(1 for r, s in my_hand if r in HIGH_RANKS)

    if high_card_count == 0:
        return lambda c: c[0] in HIGH_RANKS

    elif high_card_count == 1:
        if pick_number == 4:
            return lambda c: True  # random for pick 4
        else:  # pick 5 — check if pick 4 was a high card
            # my_hand now has 4 cards, check if the 4th was K/A
            pick_4_was_high = my_hand[3][0] in HIGH_RANKS
            if pick_4_was_high:
                return lambda c: True  # now have 2 high cards, go random
            else:
                return lambda c: c[0] in HIGH_RANKS  # still only 1, target K/A

    else:
        return lambda c: True

# ---- Single game ----

def play_game(n_picks=5, dealer_fill=8):
    my_hand = []
    dealer_hand = []
    remaining_deck = list(DECK)

    for pick_number in range(1, n_picks + 1):
        if not remaining_deck:
            break
        rule_fn = get_rule(pick_number, my_hand)
        matching = [c for c in remaining_deck if rule_fn(c)]
        if not matching:
            break
        random.shuffle(remaining_deck)
        target_idx = next(i for i, c in enumerate(remaining_deck) if rule_fn(c))
        my_hand.append(remaining_deck[target_idx])
        dealer_hand += remaining_deck[:target_idx]
        remaining_deck = remaining_deck[target_idx + 1:]

    if len(my_hand) < 5:
        return 0, 0, 0

    dealer_size_before_fill = len(dealer_hand)
    random.shuffle(remaining_deck)
    needed = max(0, dealer_fill - len(dealer_hand))
    final_dealer = dealer_hand + remaining_deck[:needed]
    dealer_size_after_fill = len(final_dealer)

    win = 1 if best_hand_rank(my_hand) > best_hand_rank(final_dealer) else 0
    return win, dealer_size_before_fill, dealer_size_after_fill

# ---- Simulate ----

def simulate(n_games=5000, n_picks=5, dealer_fill=8):
    wins = 0
    total_before = 0
    total_after = 0
    for _ in range(n_games):
        win, before, after = play_game(n_picks, dealer_fill)
        wins += win
        total_before += before
        total_after += after
    return wins / n_games, total_before / n_games, total_after / n_games

if __name__ == '__main__':
    n_games = 5000
    print("Strategy:")
    print("  Picks 1-3: any card")
    print("  If 0 high cards (K/A) after pick 3: picks 4+5 target K/A only")
    print("  If 1 high card after pick 3:")
    print("    pick 4 random")
    print("    if pick 4 was K/A -> pick 5 random")
    print("    if pick 4 was not K/A -> pick 5 targets K/A")
    print("  If 2+ high cards after pick 3: picks 4+5 random")
    print(f"\nSimulating over {n_games} games...\n")
    win_rate, avg_before, avg_after = simulate(n_games=n_games)
    print(f"Win rate:                          {win_rate:.3f}")
    print(f"Avg dealer cards from passing:     {avg_before:.2f}")
    print(f"Avg dealer cards after fill:       {avg_after:.2f}")

Strategy:
  Picks 1-3: any card
  If 0 high cards (K/A) after pick 3: picks 4+5 target K/A only
  If 1 high card after pick 3:
    pick 4 random
    if pick 4 was K/A -> pick 5 random
    if pick 4 was not K/A -> pick 5 targets K/A
  If 2+ high cards after pick 3: picks 4+5 random

Simulating over 5000 games...

Win rate:                          0.151
Avg dealer cards from passing:     4.30
Avg dealer cards after fill:       8.81
